## Test notebook for auto bining

In [30]:
import os
import pandas as pd
import numpy as np

In [31]:
#read data from data/PKV RED REMARKS.xlsm
path = os.path.join('..', 'data', 'PKV RED REMARKS.xlsm')
print(path)
data = pd.read_excel(path, sheet_name='Sorted Issues')
bins = pd.read_excel(path, sheet_name='General Bin List')
print("Data loaded successfully")

..\data\PKV RED REMARKS.xlsm
Data loaded successfully


In [32]:
bins_dict = {col: bins[col].dropna().tolist() for col in bins.columns}
print(bins_dict)
print(bins_dict.keys())

{'Unassigned': [], 'As Intended': [], '12V system problem': ['12V drained', 'Battery cable', 'CEM B11D887 LIN BMS', 'Critical charging message', 'DIM message 12v system service required', 'Low Voltage system message', 'Message: 12V battery cannot be charged', 'Multiple warning messages', 'Redundant battery front, low SOC', 'Steering wheel button, light on', 'Support battery'], '12V system problem - WISE': [], '3:rd party APP': ['Air quality app', 'Amazon Prime App', 'App-drive mode availability', 'Apple Music', 'Apps not available?', 'AudioWagon', 'B&W Experience app', 'Dolby atmos ', 'EasyPark app', 'Game app', 'Google play issue', 'HBO App', 'Himalaya App', 'Libby app', 'MAX', 'Missing 3:rd party APP', 'QQ music', 'Spotify app', 'Spotify not possible to play/pause', 'SR app', 'Storytel app', 'Tidal app', 'Tunein app', 'Vivaldi app', 'Waze app'], 'AC': ['Noise', 'Other', 'Performance', 'System leakage', 'Vibrations'], 'AC charge issue': [], 'ACC': ['ACC - Driver assistance system book

In [33]:
import numpy as np
from sklearn.model_selection import train_test_split
#create new df from "data" containing only columns "Summary" and "Solution"
filtered_data = data[data['BIN'].isin(list(bins_dict.keys())[1:])]
# features = filtered_data[['Summary', 'Solution']]
# target = filtered_data[['BIN', 'BIN Specific']]
# The original 'data' remains unchanged after this cell

print(filtered_data.shape)
# Shuffle the rows
filtered_data = filtered_data.sample(frac=1, random_state=42).reset_index(drop=True)

# Split into train and test sets (e.g., 80% train, 20% test)
train_data, test_data = train_test_split(filtered_data, test_size=0.15, random_state=42)

features_train = train_data[['Summary', 'Solution']]
target_train = train_data[['BIN', 'BIN Specific']]
features_test = test_data[['Summary', 'Solution']]
target_test = test_data[['BIN', 'BIN Specific']]

(2857, 19)


In [34]:
print(features_train.head())
print(target_train.head())

                                                Summary  \
1360                  Dim message nödsamtal fungerar ej   
736   RCN44J/XLP1191 - Unable to stop the restore of...   
96    AGX28J/XLP1177 - Contact name is not shown dur...   
2428  Ongoing call still shown a moment after ending...   
2125       Suspension fault in DIM at new drive cycle.    

                     Solution  
1360     Connected Experience  
736      Connected Experience  
96       Connected Experience  
2428     Connected Experience  
2125  Safe Vehicle Automation  
                                       BIN          BIN Specific
1360                         E-call issues    E-call malfunction
736   Driver profile and personal settings  Other profile issues
96                               Bluetooth            Phone book
2428                          Graphic bugs          Graphic bugs
2125                        Air suspension           DIM message


In [35]:
OLD_BIN_ALIASES: dict[str, str] = {
    "acc": "ACC",
    "adas": "General Intellisafe Problems",
    "ahbc": "Automatic high beam",
    "alarm": "Alarm",
    "app": "Mobile App problem",
    # "battery & charging": "No charging",
    "blis": "BLIS",
    "bluetooth": "Bluetooth",
    # "brake lights": "Rear lamps & lighting",
    "climate": "General climate",
    # "collision avoidance": "Collision Avoidance",
    "csa": "Curve Speed Assist",
    "csd": "CSD",
    # "csd & dim": "CSD and DIM black",
    # "digital key": "Key Not Found",
    "dim": "DIM",
    # "dim messages": "DIM",
    "display": "Display low brightness/dark",
    "dms": "Driver Monitoring System (DMS)",
    # "doors & locks": "Door lock",
    # "driveline": "Driveline issue",
    "esc": "Electronic Stability Control (ESC)",
    # "factory reset": "Driver profile and personal settings",
    # "google": "Navigation System Problem",
    # "google assistant": "Gemini Voice assistant",
    # "google maps": "Navigation System Problem",
    # "gps": "Navigation System Problem",
    # "graphical bug": "Graphic bugs",
    # "hardware": "Other",
    # "hazard warning": "DIM",
    "hod": "Hands off Detection (HOD) system",
    "isa": "ISA",
    # "key fob": "Key Fob HW related",
    "lca": "LCA - Lane Change Assist",
    "ldw": "LDW",
    # "lights": "Headlamps",
    "lka": "LKA",
    # "media & sound": "Audio Sound Problem",
    "mirrors": "Outer rear view mirrors",
    "navigation": "Navigation System Problem",
    "nvh": "Driveline vibrations",
    "opr": "OPR",
    "other": "Other",
    "pa": "Pilot Assist",
    "pac": "PAC 360",
    # "phone": "Mobile App problem",
    # "pilot assist": "Pilot Assist",
    "pot": "POT",
    "profile": "Driver profile and personal settings",
    # "radio": "Radio problem",
    # "rain sensor": "Rain sensor",
    "rsi": "RSI",
    "seatbelt": "Seatbelt front",
    "seats": "Seat front",
    "sla": "Slippery Road",
    "soc": "Range info",
    "sos/e-call": "E-call issues",
    # "spotify": "Audio Sound Problem",
    # "steering wheel": "Steering wheel",
    # "suspension": "Suspension",
    "tpms": "TPMS",
    # "unassigned": "Other",
    # "unknown": "Other",
    # "as intended": "Other",
    # "n/a": "Other",
    # "conceptual": "Other",
    # "widget": "Driver & Vehicle Info",
    # "windows & wipers": "Front wipers",
    "wpc": "Wireless Phone Charger",
}

def replace_words_in_text(text, mapping):
    words = text.lower().split()
    replaced = [mapping.get(word, word) for word in words]
    return ' '.join(replaced)

def formated_features(features):
    features = features.fillna('empty')
    features['Summary'] = features['Summary'].apply(lambda x: replace_words_in_text(x, OLD_BIN_ALIASES))
    features['Summary'] = features['Summary'].apply(lambda x: x.rsplit("-", 1)[-1])
    # print(features['Summary'].iloc[150:200])
    # features = features.astype(str).str.lower()
    features = features['Solution'].astype(str) + ' | ' + features['Summary'].astype(str) 
    # features = features['Summary'].astype(str) #TODO: with or without solution?

    features = features.str.lower()
    return features

def formated_target(target):
    target = target.fillna('')
    target = (target['BIN'].astype(str) + ' | ' + target['BIN Specific'].astype(str)).str.strip()
    return target

features_train = formated_features(features_train)
target_train = formated_target(target_train)
features_test = formated_features(features_test)
target_test = formated_target(target_test)


# print(features.iloc[100:150])
# print(target.head())
# print(features.shape, target.shape)

### IF-IDF embedding

In [47]:
from sklearn.feature_extraction.text import TfidfVectorizer


vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=2) #TODO: tune ngram range and min_df
mat_features = vectorizer.fit_transform(features_train)

print(vectorizer.get_feature_names_out())
print(mat_features.toarray())
print(mat_features.toarray().shape)

['10' '10 min' '100' ... 'youtube' 'zone' 'är']
[[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]
(2428, 3559)


### e5-small

In [37]:
# import torch.nn.functional as F

# from torch import Tensor
# from transformers import AutoTokenizer, AutoModel

from sentence_transformers import SentenceTransformer

print(features_train.head())

# prefix = "query: "
# features_train = features_train.apply(lambda x: prefix + x)
# features_test = features_test.apply(lambda x: prefix + x)

# def add_prefix(df, prefix="query: "):
def add_prefix(df, prefix="query: "):
    return df.apply(lambda x: prefix + x)

model = SentenceTransformer('intfloat/multilingual-e5-small')
embeddings = model.encode(add_prefix(features_train, prefix="passage: ").tolist(), normalize_embeddings=True)


print(embeddings.shape)


1360    connected experience | dim message nödsamtal f...
736     connected experience |  unable to stop the res...
96      connected experience |  contact name is not sh...
2428    connected experience | ongoing call still show...
2125    safe vehicle automation | suspension fault in ...
dtype: str
(2428, 384)


### init KNN

In [48]:
from sklearn.neighbors import KNeighborsClassifier

K = 20

def init_KNN(mat):
    knn = KNeighborsClassifier(
        n_neighbors=K,
        metric="cosine",
        weights="distance"
    )
    # knn.fit(mat_features, target_train['BIN'])
    knn.fit(mat, target_train)
    return knn

knn_IFIDF = init_KNN(mat_features)
knn_embed = init_KNN(embeddings)

### Examples

In [39]:

def predict_samples(knn, vect_func, features_test, target_train, target_test):
    for i in range(0, 5):
        sample = features_test.iloc[i]
        sample_vector = vect_func([sample])
        distances, indices = knn.kneighbors(sample_vector)
        weights = 1 - distances

        print("Sample: ", sample)
        # print("Predicted BIN:", target_train['BIN'].iloc[indices[0][0]])
        # print("Actual BIN:", target_test['BIN'].iloc[i])
        print("Predicted BIN (k=1)", "(d=)", round(weights[0][0], 2), ":", target_train.iloc[indices[0][0]])
        # print("Predicted BIN (k=2)", "(d=)", round(weights[0][1], 2), ":", target_train.iloc[indices[0][1]])
        print("Predicted BIN (k=3)", "(d=)", round(weights[0][2], 2), ":", target_train.iloc[indices[0][2]])
        print("Predicted BIN (k=4)", "(d=)", round(weights[0][3], 2), ":", target_train.iloc[indices[0][3]])
        print("Predicted BIN (k=5)", "(d=)", round(weights[0][4], 2), ":", target_train.iloc[indices[0][4]])
        print("Actual BIN:", target_test.iloc[i])
        print()

print("KNN with TF-IDF:")
predict_samples(knn_IFIDF, vectorizer.transform, features_test, target_train, target_test)
print("KNN with Embeddings:")
predict_samples(knn_embed, lambda x: model.encode(x, normalize_embeddings=True), add_prefix(features_test, prefix="query: "), target_train, target_test)

KNN with TF-IDF:
Sample:  connected experience |  music through spotify mobile app problem pauses by it self.
Predicted BIN (k=1) (d=) 0.42 : Door complete, front | Other
Predicted BIN (k=3) (d=) 0.39 : 3:rd party APP | Spotify app
Predicted BIN (k=4) (d=) 0.37 : Audio Sound Problem | Sound setting
Predicted BIN (k=5) (d=) 0.36 : Bluetooth | BT no media sound
Actual BIN: 3:rd party APP | Spotify not possible to play/pause

Sample:  adadas | blis didn´t deactivate after it had been triggered
Predicted BIN (k=1) (d=) 0.35 : BLIS | BLIS False alerts
Predicted BIN (k=3) (d=) 0.26 : One pedal drive | OPD - Settings
Predicted BIN (k=4) (d=) 0.26 : Panoroof | Function
Predicted BIN (k=5) (d=) 0.24 : BLIS | BLIS False alerts
Actual BIN: BLIS | BLIS No alert

Sample:  connected experience | gps positioning all over the place during drive cycle.
Predicted BIN (k=1) (d=) 0.43 : DIM Black | Black Permanently
Predicted BIN (k=3) (d=) 0.3 : Infotainment Rebooting | Reboot
Predicted BIN (k=4) (d=) 0.

### Performance

In [40]:
xs = np.arange(0.6, 1, 0.02)
print(xs)
print([round(float(x**2), 2) for x in xs])
print([round(float(x**3), 2) for x in xs])
print([round(float(x**4), 2) for x in xs])

[0.6  0.62 0.64 0.66 0.68 0.7  0.72 0.74 0.76 0.78 0.8  0.82 0.84 0.86
 0.88 0.9  0.92 0.94 0.96 0.98]
[0.36, 0.38, 0.41, 0.44, 0.46, 0.49, 0.52, 0.55, 0.58, 0.61, 0.64, 0.67, 0.71, 0.74, 0.77, 0.81, 0.85, 0.88, 0.92, 0.96]
[0.22, 0.24, 0.26, 0.29, 0.31, 0.34, 0.37, 0.41, 0.44, 0.47, 0.51, 0.55, 0.59, 0.64, 0.68, 0.73, 0.78, 0.83, 0.88, 0.94]
[0.13, 0.15, 0.17, 0.19, 0.21, 0.24, 0.27, 0.3, 0.33, 0.37, 0.41, 0.45, 0.5, 0.55, 0.6, 0.66, 0.72, 0.78, 0.85, 0.92]


In [49]:

from collections import Counter

total_predictions = features_test.shape[0]

def find_most_common(els):
    counter = Counter(els)
    max_count = max(counter.values())
    for el in els:
        if counter[el] == max_count:
            return el
        
def find_weighted_most_common(indices, weights, target_train, weight_func=lambda x: x**2):
    labels = target_train.iloc[indices]
    labels_weights = {label: 0 for label in labels}
    
    for label, weight in zip(labels, weights):
        labels_weights[label] += weight_func(weight)

    return max(labels_weights, key=labels_weights.get)


def predict_and_evaluate(knn, vect_func, ks_settings, features_test, target_train, target_test):
    ks = ks_settings.keys()
    theresholds = {k: ks_settings[k] for k in ks}
    corrects = {k: 0 for k in ks}
    incorrects = {k: 0 for k in ks}
    skips_lowconf = {k: 0 for k in ks}
    skips_closecall = {k: 0 for k in ks} #TODO: define close call
    cor_dist = {k: [] for k in ks}
    inc_dist = {k: [] for k in ks}

    for i in range(features_test.shape[0]):
        sample = features_test.iloc[i]
        sample_vector = vect_func([sample])
        distances, indices = knn.kneighbors(sample_vector)
        weights = 1 - distances
        # print(distances)

        for k in ks:
            if weights[0][0] < theresholds[k]: #catch low confidence predictions
                skips_lowconf[k] += 1
            elif find_weighted_most_common(indices[0][0:k], weights[0][0:k], target_train) == target_test.iloc[i]:
                corrects[k] += 1
                cor_dist[k].append(weights[0][0])
            else:
                incorrects[k] += 1
                inc_dist[k].append(weights[0][0])

    # print(f"Correct predictions for TF-IDF + KNN")
    print(f"Total predictions: {total_predictions}")

    for k in ks:
        print(f"Accuracy for k={k}: \t\t{corrects[k] / total_predictions:.2f}")
        if k >= 10:
            print(f"Skipped preds for k={k}: \t{skips_lowconf[k] / total_predictions:.2f}")
        else:
            print(f"Skipped preds for k={k}: \t\t{skips_lowconf[k] / total_predictions:.2f}")
        print(f"Incorrect preds for k={k}: \t{incorrects[k] / total_predictions:.2f}\n")
        
    # print(f"Accuracy for k=2: {k2_correct_predictions / total_predictions:.2f}")
    # print(f"Accuracy for k=3: {k3_correct_predictions / total_predictions:.2f}")
    # print(f"Accuracy for k=5: {k5_correct_predictions / total_predictions:.2f}")

    print("all")
    # print(corrects.values())
    print(f"Distance stats correct predictions: Mean {np.mean(sum(list(cor_dist.values()),[])):.2f}, Max {max(sum(list(cor_dist.values()),[])):.2f}, Min {min(sum(list(cor_dist.values()),[])):.2f}")
    print(f"Distance stats incorrect predictions: Mean {np.mean(sum(inc_dist.values(),[])):.2f}, Max {max(sum(inc_dist.values(),[])):.2f}, Min {min(sum(inc_dist.values(),[])):.2f}")

    for k in ks:
        if not cor_dist[k] or not inc_dist[k]: #is empty
            print("all skipped?")
            cor_dist[k] = [-1]
            inc_dist[k] = [-1]
        print(f"k{k}")
        print(f"Distance stats correct predictions: Mean {np.mean(cor_dist[k]):.2f}, Max {max(cor_dist[k]):.2f}, Min {min(cor_dist[k]):.2f}")
        print(f"Distance stats incorrect predictions: Mean {np.mean(inc_dist[k]):.2f}, Max {max(inc_dist[k]):.2f}, Min {min(inc_dist[k]):.2f}")




### No Threshold

In [50]:
KS_settings = {1: 0.0, 3: 0.0, 5: 0.0, 10: 0.0}
print("-"*20 + " IF-IDF" + "-"*20)
predict_and_evaluate(knn_IFIDF, vectorizer.transform, KS_settings, features_test, target_train, target_test)


-------------------- IF-IDF--------------------
Total predictions: 429
Accuracy for k=1: 		0.41
Skipped preds for k=1: 		0.00
Incorrect preds for k=1: 	0.59

Accuracy for k=3: 		0.45
Skipped preds for k=3: 		0.00
Incorrect preds for k=3: 	0.55

Accuracy for k=5: 		0.48
Skipped preds for k=5: 		0.00
Incorrect preds for k=5: 	0.52

Accuracy for k=10: 		0.49
Skipped preds for k=10: 	0.00
Incorrect preds for k=10: 	0.51

all
Distance stats correct predictions: Mean 0.70, Max 1.00, Min 0.26
Distance stats incorrect predictions: Mean 0.55, Max 1.00, Min 0.21
k1
Distance stats correct predictions: Mean 0.71, Max 1.00, Min 0.26
Distance stats incorrect predictions: Mean 0.55, Max 1.00, Min 0.21
k3
Distance stats correct predictions: Mean 0.70, Max 1.00, Min 0.26
Distance stats incorrect predictions: Mean 0.55, Max 1.00, Min 0.21
k5
Distance stats correct predictions: Mean 0.69, Max 1.00, Min 0.29
Distance stats incorrect predictions: Mean 0.55, Max 1.00, Min 0.21
k10
Distance stats correct pre

In [43]:
KS_settings = {1: 0.0, 3: 0.0, 5: 0.0, 10: 0.0, 15: 0.0, 20: 0.0}
print("-"*20 + " Embedding" + "-"*20)
predict_and_evaluate(knn_embed, lambda x: model.encode(x, normalize_embeddings=True), KS_settings, add_prefix(features_test), target_train, target_test)

-------------------- Embedding--------------------
Total predictions: 429
Accuracy for k=1: 		0.52
Skipped preds for k=1: 		0.00
Incorrect preds for k=1: 	0.48

Accuracy for k=3: 		0.52
Skipped preds for k=3: 		0.00
Incorrect preds for k=3: 	0.48

Accuracy for k=5: 		0.53
Skipped preds for k=5: 		0.00
Incorrect preds for k=5: 	0.47

Accuracy for k=10: 		0.53
Skipped preds for k=10: 	0.00
Incorrect preds for k=10: 	0.47

Accuracy for k=15: 		0.50
Skipped preds for k=15: 	0.00
Incorrect preds for k=15: 	0.50

Accuracy for k=20: 		0.48
Skipped preds for k=20: 	0.00
Incorrect preds for k=20: 	0.52

all
Distance stats correct predictions: Mean 0.92, Max 0.96, Min 0.87
Distance stats incorrect predictions: Mean 0.91, Max 0.96, Min 0.85
k1
Distance stats correct predictions: Mean 0.92, Max 0.96, Min 0.87
Distance stats incorrect predictions: Mean 0.91, Max 0.96, Min 0.85
k3
Distance stats correct predictions: Mean 0.92, Max 0.96, Min 0.87
Distance stats incorrect predictions: Mean 0.91, Max 0

### With Threshold

In [51]:

KS_settings = {1: 0.70, 3: 0.60, 5: 0.60, 10: 0.60}
print("-"*20 + " IF-IDF" + "-"*20)
predict_and_evaluate(knn_IFIDF, vectorizer.transform, KS_settings, features_test, target_train, target_test)


-------------------- IF-IDF--------------------
Total predictions: 429
Accuracy for k=1: 		0.21
Skipped preds for k=1: 		0.69
Incorrect preds for k=1: 	0.10

Accuracy for k=3: 		0.30
Skipped preds for k=3: 		0.54
Incorrect preds for k=3: 	0.16

Accuracy for k=5: 		0.31
Skipped preds for k=5: 		0.54
Incorrect preds for k=5: 	0.15

Accuracy for k=10: 		0.30
Skipped preds for k=10: 	0.54
Incorrect preds for k=10: 	0.16

all
Distance stats correct predictions: Mean 0.83, Max 1.00, Min 0.60
Distance stats incorrect predictions: Mean 0.79, Max 1.00, Min 0.60
k1
Distance stats correct predictions: Mean 0.88, Max 1.00, Min 0.70
Distance stats incorrect predictions: Mean 0.86, Max 1.00, Min 0.70
k3
Distance stats correct predictions: Mean 0.81, Max 1.00, Min 0.60
Distance stats incorrect predictions: Mean 0.77, Max 1.00, Min 0.61
k5
Distance stats correct predictions: Mean 0.81, Max 1.00, Min 0.60
Distance stats incorrect predictions: Mean 0.77, Max 1.00, Min 0.60
k10
Distance stats correct pre

In [45]:
KS_settings = {1: 0.94, 3: 0.94, 5: 0.93, 10: 0.93, 15: 0.93, 20: 0.93}
print("-"*20 + " Embedding" + "-"*20)
predict_and_evaluate(knn_embed, lambda x: model.encode(x, normalize_embeddings=True), KS_settings, add_prefix(features_test), target_train, target_test)

# KS_settings = {1: 0.0, 3: 0.0, 5: 0.0, 10: 0.0, 15: 0.0, 20: 0.0}
# print("-"*20 + " Embedding" + "-"*20)
# predict_and_evaluate(knn_embed, lambda x: model.encode(x, normalize_embeddings=True), KS_settings, add_prefix(features_test), target_train, target_test)

-------------------- Embedding--------------------
Total predictions: 429
Accuracy for k=1: 		0.12
Skipped preds for k=1: 		0.83
Incorrect preds for k=1: 	0.04

Accuracy for k=3: 		0.12
Skipped preds for k=3: 		0.83
Incorrect preds for k=3: 	0.04

Accuracy for k=5: 		0.23
Skipped preds for k=5: 		0.69
Incorrect preds for k=5: 	0.08

Accuracy for k=10: 		0.22
Skipped preds for k=10: 	0.69
Incorrect preds for k=10: 	0.09

Accuracy for k=15: 		0.21
Skipped preds for k=15: 	0.69
Incorrect preds for k=15: 	0.10

Accuracy for k=20: 		0.20
Skipped preds for k=20: 	0.69
Incorrect preds for k=20: 	0.11

all
Distance stats correct predictions: Mean 0.94, Max 0.96, Min 0.93
Distance stats incorrect predictions: Mean 0.94, Max 0.96, Min 0.93
k1
Distance stats correct predictions: Mean 0.95, Max 0.96, Min 0.94
Distance stats incorrect predictions: Mean 0.95, Max 0.96, Min 0.94
k3
Distance stats correct predictions: Mean 0.95, Max 0.96, Min 0.94
Distance stats incorrect predictions: Mean 0.95, Max 0

In [46]:
# #add distance thershold

# DIST_THERESHOLD_K1 = 0.96
# DIST_THERESHOLD_K3 = 0.95
# DIST_THERESHOLD_K5 = 0.94

# total_predictions = features_test.shape[0]
# k1_correct_predictions = 0
# k3_correct_predictions = 0
# k5_correct_predictions = 0
# k1_incorrect_predictions = 0
# k3_incorrect_predictions = 0
# k5_incorrect_predictions = 0
# k1_skiped_predictions = 0
# k3_skiped_predictions = 0
# k5_skiped_predictions = 0

# for i in range(features_test.shape[0]):
#     sample = features_test.iloc[i]
#     sample_vector = model.encode([sample], normalize_embeddings=True)
#     distances, indices = knn.kneighbors(sample_vector)
#     distances = 1 - distances

#     #k=1
#     if distances[0][0] < DIST_THERESHOLD_K1:
#         k1_skiped_predictions += 1
#     elif target_train.iloc[indices[0][0]] == target_test.iloc[i]:
#         k1_correct_predictions += 1
#     else:
#         k1_incorrect_predictions += 1

#     # #k=2
#     # if find_most_common(target_train.iloc[indices[0][0:2]]) == target_test.iloc[i]:
#     #     k2_correct_predictions += 1

#     #k=3
#     if np.mean(distances[0][0:3]) < DIST_THERESHOLD_K3:
#         k3_skiped_predictions += 1
#     elif find_most_common(target_train.iloc[indices[0][0:3]]) == target_test.iloc[i]:
#         k3_correct_predictions += 1
#     else:
#         k3_incorrect_predictions += 1

#     #k=5
#     # print()
#     # print(target_train.iloc[indices[0][0]])
#     # print(target_train.iloc[indices[0][0:2]])
#     # print(target_test.iloc[i])
#     if np.mean(distances[0][0:5]) < DIST_THERESHOLD_K5:
#         k5_skiped_predictions += 1
#     elif find_most_common(target_train.iloc[indices[0][0:5]]) == target_test.iloc[i]:
#         k5_correct_predictions += 1
#     else:
#         k5_incorrect_predictions += 1



# print(f"Correct predictions for TF-IDF + KNN")
# print(f"Total predictions: {total_predictions}")
# print(f"Accuracy for k=1: \t\t{k1_correct_predictions / total_predictions:.2f}")
# print(f"Skipped predictions for k=1: \t{k1_skiped_predictions/ total_predictions:.2f}")
# print(f"Incorrect predictions for k=1: \t{k1_incorrect_predictions / total_predictions:.2f}\n")

# print(f"Accuracy for k=3: \t\t{k3_correct_predictions / total_predictions:.2f}")
# print(f"Skipped predictions for k=3: \t{k3_skiped_predictions/ total_predictions:.2f}")
# print(f"Incorrect predictions for k=3: \t{k3_incorrect_predictions / total_predictions:.2f}\n")

# print(f"Accuracy for k=5: \t\t{k5_correct_predictions / total_predictions:.2f}")
# print(f"Skipped predictions for k=5: \t{k5_skiped_predictions/ total_predictions:.2f}")
# print(f"Incorrect predictions for k=5: \t{k5_incorrect_predictions / total_predictions:.2f}\n")